## 1. Connect and prepare folders
Connects to PostgreSQL and creates the `data/` folder where generated files are saved. `data/` is in `.gitignore`, so it is never pushed to GitHub.

In [12]:
# --- Imports ---
import os                                   # file paths and environment variables
from dotenv import load_dotenv              # reads settings from the .env file
import psycopg                              # PostgreSQL driver

# --- Load the database URL and connect (same pattern as notebook 01) ---
load_dotenv(os.path.expanduser("~/bank-rag-guard/.env"))
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

# --- Create the data folder if it does not exist yet ---
DATA_DIR = os.path.expanduser("~/bank-rag-guard/data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Connected. Data folder:", DATA_DIR)

Connected. Data folder: /home/linux1/bank-rag-guard/data


## 2. Helpers that generate valid-looking PII
- `luhn_check_digit`: computes the final check digit of an Emirates ID
- `make_emirates_id`: format `784-YYYY-XXXXXXX-Z`
- `make_uae_iban`: 23-character IBAN starting with `AE`, with correct mod-97 check digits
- `to_arabic_digits`: converts 0-9 to Arabic-Indic digits, to test the masker on Arabic-written numbers

In [13]:
import random
import re

SEED = 42            # fixed seed so every run generates the same data
random.seed(SEED)

def luhn_check_digit(digits: str) -> str:
    """Return the Luhn check digit for a string of digits."""
    total = 0
    # Walk from the rightmost digit; double every second digit starting with the first
    for i, ch in enumerate(reversed(digits)):
        d = int(ch)
        if i % 2 == 0:
            d *= 2
            if d > 9:
                d -= 9               # same as adding the two digits of the product
        total += d
    return str((10 - total % 10) % 10)

def make_emirates_id() -> str:
    """Build an ID in the format 784-YYYY-XXXXXXX-Z."""
    year = random.randint(1950, 2005)               # birth year part
    serial = f"{random.randint(0, 9_999_999):07d}"  # 7 random digits
    check = luhn_check_digit("784" + str(year) + serial)
    return f"784-{year}-{serial}-{check}"

def make_uae_iban() -> str:
    """Build a 23-character UAE IBAN: AE + 2 check digits + 3-digit bank + 16-digit account."""
    bank = f"{random.randint(0, 999):03d}"
    account = f"{random.randint(0, 10**16 - 1):016d}"
    bban = bank + account
    # IBAN check digits: append "AE00" as numbers (A=10, E=14), then 98 - (value mod 97)
    check = 98 - int(bban + "101400") % 97
    return f"AE{check:02d}{bban}"

def make_phone() -> str:
    """UAE mobile number, e.g. +971501234567."""
    return "+9715" + f"{random.randint(0, 99_999_999):08d}"

def to_arabic_digits(text: str) -> str:
    """Replace 0-9 with Arabic-Indic digits (٠-٩)."""
    return text.translate(str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩"))

# --- Quick self-test of the generators ---
sample_eid, sample_iban = make_emirates_id(), make_uae_iban()
assert re.fullmatch(r"784-\d{4}-\d{7}-\d", sample_eid)
assert len(sample_iban) == 23 and sample_iban.startswith("AE")
print(sample_eid, sample_iban, make_phone())

784-1990-1867825-2 AE110254954488479039739 +971532868828


## 3. Generate synthetic customer records
Creates 300 fake customers with English or Arabic names, plus a short case note that embeds their Emirates ID, IBAN and phone number. About a third of the Arabic notes write numbers in Arabic-Indic digits. The raw fields are kept as **ground truth** so we can later measure how well the masker catches them. Each record gets a clearance tier (1 = most common, 3 = most restricted).

In [14]:
import pandas as pd
from faker import Faker

Faker.seed(SEED)
fake_en = Faker("en_US")       # English names
fake_ar = Faker("ar_AA")       # Arabic names (Arabic script)

# --- Case-note templates; {placeholders} are filled with the fake PII ---
TEMPLATE_EN = ("Customer {name}, Emirates ID {eid}, IBAN {iban}, phone {phone}, "
               "reported a disputed card transaction and requested a review.")
TEMPLATE_AR = ("العميل {name}، رقم الهوية الإماراتية {eid}، رقم الآيبان {iban}، "
               "الهاتف {phone}، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.")

rows = []
for i in range(300):
    lang = random.choice(["en", "ar"])
    name = fake_en.name() if lang == "en" else fake_ar.name()
    eid, iban, phone = make_emirates_id(), make_uae_iban(), make_phone()

    # Fill the template; for ~1/3 of Arabic notes, write the numbers in Arabic digits
    text = (TEMPLATE_EN if lang == "en" else TEMPLATE_AR).format(
        name=name, eid=eid, iban=iban, phone=phone)
    if lang == "ar" and random.random() < 0.33:
        text = to_arabic_digits(text)

    rows.append({
        "customer_id": f"CUST{i:04d}",
        "language": lang,
        "name": name, "emirates_id": eid, "iban": iban, "phone": phone,  # ground truth
        "text": text,
        # Tier mix: 50% tier 1, 30% tier 2, 20% tier 3
        "tier": random.choices([1, 2, 3], weights=[50, 30, 20])[0],
    })

customers = pd.DataFrame(rows)
customers.to_csv(f"{DATA_DIR}/customers.csv", index=False)   # saved for later notebooks
print(customers["language"].value_counts().to_dict(), customers["tier"].value_counts().to_dict())
customers.head(4)

{'ar': 152, 'en': 148} {1: 148, 2: 100, 3: 52}


,customer_id,language,name,emirates_id,iban,phone,text,tier
0,CUST0000,en,Allison Hill,784-1958-1719583-7,AE646921566149281915956,+971579254563,"Customer Allison Hill, Emirates ID 784-1958-17...",1
1,CUST0001,en,Noah Rhodes,784-1955-3668136-4,AE952387557396729776374,+971529587039,"Customer Noah Rhodes, Emirates ID 784-1955-366...",1
2,CUST0002,ar,محمود الألجاوي,784-2001-0109031-2,AE097772876166965439421,+971593702683,العميل محمود الألجاوي، رقم الهوية الإماراتية 7...,1
3,CUST0003,en,Daniel Wagner,784-1998-5647119-0,AE401046844159633535249,+971512981052,"Customer Daniel Wagner, Emirates ID 784-1998-5...",1


## 4. Create demo users
Adds one account per role: a teller (tier 1), a compliance officer (tier 3) and an auditor. These are used to prove that the same question gives different answers depending on clearance.

In [15]:
# --- (username, role, tier) for each demo account ---
demo_users = [
    ("teller1",     "teller",     1),   # branch staff: lowest clearance
    ("compliance1", "compliance", 3),   # compliance officer: highest clearance
    ("auditor1",    "auditor",    1),   # auditor: checks logs, not documents
]

for user in demo_users:
    # ON CONFLICT DO NOTHING makes this safe to re-run
    conn.execute(
        "INSERT INTO users (username, role, tier) VALUES (%s, %s, %s) "
        "ON CONFLICT (username) DO NOTHING", user)

print(conn.execute("SELECT * FROM users ORDER BY username").fetchall())

[('auditor1', 'auditor', 1), ('compliance1', 'compliance', 3), ('teller1', 'teller', 1)]


## 5. Check internet access
Tests whether the server can reach Hugging Face (dataset downloads) and the Gemini and Groq APIs. Any HTTP status number means the site is reachable; FAILED means it is blocked.

In [16]:
import requests

# --- Sites we need: dataset hub and the two LLM providers ---
sites = [
    "https://huggingface.co",
    "https://generativelanguage.googleapis.com",
    "https://api.groq.com",
]

for url in sites:
    try:
        r = requests.get(url, timeout=10)
        print(f"{url} -> reachable (HTTP {r.status_code})")   # 200/401/403/404 are all fine here
    except Exception as e:
        print(f"{url} -> FAILED ({type(e).__name__})")

https://huggingface.co -> reachable (HTTP 200)
https://generativelanguage.googleapis.com -> reachable (HTTP 404)
https://api.groq.com -> reachable (HTTP 200)


## 6. Helper: read Hugging Face dataset rows through the web API
Hugging Face's `datasets-server` API returns dataset rows as JSON, 100 per request, so we don't need `pyarrow` (which can't be installed on this server's architecture). `hf_rows` finds the dataset's config name, then pages through the rows.

In [17]:
import os, io, json, time, zipfile
import requests
import pandas as pd

HF_ROWS = "https://datasets-server.huggingface.co"

def hf_rows(dataset, split, n):
    """Return up to n rows (list of dicts) from a Hugging Face dataset split, via the REST API."""
    # --- Step 1: look up the config name for this split (usually "default") ---
    info = requests.get(f"{HF_ROWS}/splits", params={"dataset": dataset}, timeout=30)
    info.raise_for_status()
    matches = [s for s in info.json()["splits"] if s["split"] == split]
    if not matches:
        available = [s["split"] for s in info.json()["splits"]]
        raise ValueError(f"No split '{split}' in {dataset}. Available: {available}")
    config = matches[0]["config"]

    rows, target = [], n
    # --- Step 2: fetch up to 100 rows per request until we have enough ---
    while len(rows) < target:
        length = min(100, target - len(rows))          # 100 is the API's maximum
        for attempt in range(5):                       # retry briefly if the service is busy
            r = requests.get(f"{HF_ROWS}/rows", timeout=60, params={
                "dataset": dataset, "config": config, "split": split,
                "offset": len(rows), "length": length})
            if r.status_code == 200:
                break
            time.sleep(3 * (attempt + 1))
        r.raise_for_status()
        payload = r.json()
        target = min(n, payload.get("num_rows_total", n))   # never ask beyond the dataset's size
        batch = [item["row"] for item in payload["rows"]]
        if not batch:
            break
        rows.extend(batch)

    print(f"OK   {dataset} [{config}/{split}]: {len(rows)} rows")
    return rows

## 7. Prompt-injection dataset (attacker seeds)
Downloads `deepset/prompt-injections` (label 1 = injection, 0 = normal prompt) and saves it for the adversarial loop.

In [18]:
# --- Download both splits (the dataset is small, so we ask for up to 1,000 rows each) ---
pi_train = pd.DataFrame(hf_rows("deepset/prompt-injections", "train", 1000))
pi_test = pd.DataFrame(hf_rows("deepset/prompt-injections", "test", 1000))

# --- Save for later notebooks ---
pi_train.to_csv(f"{DATA_DIR}/prompt_injections_train.csv", index=False)
pi_test.to_csv(f"{DATA_DIR}/prompt_injections_test.csv", index=False)
print(len(pi_train), "train rows,", len(pi_test), "test rows")
print(pi_train["label"].value_counts().to_dict())   # injections vs normal prompts

OK   deepset/prompt-injections [default/train]: 546 rows
OK   deepset/prompt-injections [default/test]: 116 rows
546 train rows, 116 test rows
{0: 343, 1: 203}


## 8. PII benchmark (ai4privacy)
Downloads 1,000 labelled examples with known PII positions. Later we run our masker on these and measure recall and precision. The dataset has no Arabic, which is why we generated our own Arabic data in Cell 3.

In [19]:
# --- Try the 300k dataset first; fall back to the 200k one if that fails ---
try:
    pii_rows = hf_rows("ai4privacy/pii-masking-300k", "validation", 1000)
except Exception as e:
    print("300k failed:", type(e).__name__, str(e)[:120])
    pii_rows = hf_rows("ai4privacy/pii-masking-200k", "train", 1000)

pii_bench = pd.DataFrame(pii_rows)
# JSON Lines keeps the nested PII labels intact (CSV would flatten them badly)
pii_bench.to_json(f"{DATA_DIR}/pii_benchmark.jsonl", orient="records", lines=True, force_ascii=False)

print(pii_bench.columns.tolist())                  # which fields the dataset provides
print(pii_bench.iloc[0]["source_text"][:200])      # peek at one example

300k failed: HTTPError 500 Server Error: Internal Server Error for url: https://datasets-server.huggingface.co/rows?dataset=ai4privacy%2Fpii-ma
OK   ai4privacy/pii-masking-200k [default/train]: 1000 rows
['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?


## 9. Legal contracts (CUAD)
Downloads the official CUAD release (a zip file on GitHub), reads the contracts out of it, and keeps one row per contract. These become our Tier 2/3 documents.

In [20]:
# --- Download the official CUAD release (a zip file; roughly tens of MB) ---
CUAD_URL = "https://github.com/TheAtticusProject/cuad/raw/main/data.zip"
resp = requests.get(CUAD_URL, timeout=300)
resp.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(resp.content))     # open the zip in memory

# --- Pick the main JSON file (CUADv1.json), or the largest JSON file if the name differs ---
json_files = [name for name in zf.namelist() if name.endswith(".json")]
print("JSON files in zip:", json_files)
target = next((n for n in json_files if n.endswith("CUADv1.json")), None) \
         or max(json_files, key=lambda n: zf.getinfo(n).file_size)

# --- The file is in SQuAD format: each entry has a title and one long contract text ---
cuad = json.loads(zf.read(target))
contracts = [{"title": item["title"], "text": item["paragraphs"][0]["context"]}
             for item in cuad["data"]]

cuad_df = pd.DataFrame(contracts)
cuad_df.to_json(f"{DATA_DIR}/cuad_contracts.jsonl", orient="records", lines=True, force_ascii=False)
print(len(cuad_df), "contracts, average length", int(cuad_df["text"].str.len().mean()), "characters")

JSON files in zip: ['CUADv1.json', 'test.json', 'train_separate_questions.json']
510 contracts, average length 52563 characters


## 10. Corporate emails (Enron, original CMU release)
Streams the public CMU Enron archive (a ~1.7 GB tar.gz) and stops once we have about 1,500 usable emails, taking at most 25 per mailbox so many different people are represented. Only the part we read is downloaded, which can take a few minutes. Each email is turned into plain text: a subject line plus the body.

In [22]:
import tarfile
import email
from email import policy

ENRON_URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
TARGET_EMAILS = 1500     # stop after this many usable emails
PER_MAILBOX = 25         # at most this many from any one person's mailbox

texts, per_box = [], {}

with requests.get(ENRON_URL, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    # "r|gz" reads the archive as a stream, one file at a time, without saving the whole thing
    with tarfile.open(fileobj=resp.raw, mode="r|gz") as tar:
        for member in tar:
            if not member.isfile():
                continue
            parts = member.name.split("/")           # path looks like maildir/<mailbox>/<folder>/<file>
            if len(parts) < 4:
                continue
            box = parts[1]
            if per_box.get(box, 0) >= PER_MAILBOX:    # this mailbox is full: skip its other emails
                continue

            raw = tar.extractfile(member).read()      # must be read right away in stream mode
            try:
                msg = email.message_from_bytes(raw, policy=policy.default)
                part = msg.get_body(preferencelist=("plain",))     # prefer the plain-text part
                body = part.get_content() if part else ""
                text = f"Subject: {msg['subject'] or ''}\n\n{body}".strip()
            except Exception:
                continue                               # skip emails that cannot be parsed

            if len(text) < 200:                        # drop very short emails
                continue
            texts.append(text)
            per_box[box] = per_box.get(box, 0) + 1
            if len(texts) % 250 == 0:
                print(f"  collected {len(texts)} emails from {len(per_box)} mailboxes")
            if len(texts) >= TARGET_EMAILS:
                break                                   # leaving the loop stops the download

enron_df = pd.DataFrame({"text": texts})
enron_df.to_json(f"{DATA_DIR}/enron_emails.jsonl", orient="records", lines=True, force_ascii=False)
print(len(enron_df), "emails saved from", len(per_box), "mailboxes")
print(enron_df.iloc[0]["text"][:300])                   # peek at the first email

  collected 250 emails from 10 mailboxes
  collected 500 emails from 20 mailboxes
  collected 750 emails from 30 mailboxes
  collected 1000 emails from 40 mailboxes
  collected 1250 emails from 50 mailboxes
  collected 1500 emails from 60 mailboxes
1500 emails saved from 60 mailboxes
Subject: FW: Promotions and Transfers- Gas Logistics





Name			New Title				Effective Date			Mid Year promotion Yes/No

Floyd, Jodie		Sr Cust Svc Rep (no change)		7/16/01				No

Buehler, Craig		Sr Mkt/Sup Analyst (no change)		7/16/01				No

Wagoner, Mike		Team Advisor - Gas Control		7/1/01				No




## 11. Check saved files
Lists every file in `data/` with its size. Expect six files.

In [23]:
import os

# --- Print each saved dataset file and its size in KB ---
for name in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(os.path.join(DATA_DIR, name)) / 1024
    print(f"{name:40s} {size_kb:10.1f} KB")

cuad_contracts.jsonl                        26492.6 KB
customers.csv                                  93.4 KB
enron_emails.jsonl                           2705.3 KB
pii_benchmark.jsonl                          1483.4 KB
prompt_injections_test.csv                     14.7 KB
prompt_injections_train.csv                    65.6 KB
